In [112]:
import duckdb
print(duckdb.__version__)

con = duckdb.connect("off_canada.db") 

1.5.0


In [5]:
# count the number of products loaded
count = con.execute("SELECT COUNT(*) FROM products").fetchone()
print("Products Loaded:", count[0])

Products Loaded: 10000


In [9]:
# column names, types
cols = con.execute("DESCRIBE products").fetchall()
for col in cols:
    print(col)

('additives_n', 'INTEGER', 'YES', None, None, None)
('additives_tags', 'VARCHAR[]', 'YES', None, None, None)
('allergens_tags', 'VARCHAR[]', 'YES', None, None, None)
('brands_tags', 'VARCHAR[]', 'YES', None, None, None)
('brands', 'VARCHAR', 'YES', None, None, None)
('categories', 'VARCHAR', 'YES', None, None, None)
('categories_tags', 'VARCHAR[]', 'YES', None, None, None)
('categories_properties', 'STRUCT(ciqual_food_code INTEGER, agribalyse_food_code INTEGER, agribalyse_proxy_food_code INTEGER)', 'YES', None, None, None)
('checkers_tags', 'VARCHAR[]', 'YES', None, None, None)
('ciqual_food_name_tags', 'VARCHAR[]', 'YES', None, None, None)
('cities_tags', 'VARCHAR[]', 'YES', None, None, None)
('code', 'VARCHAR', 'YES', None, None, None)
('compared_to_category', 'VARCHAR', 'YES', None, None, None)
('complete', 'INTEGER', 'YES', None, None, None)
('completeness', 'FLOAT', 'YES', None, None, None)
('correctors_tags', 'VARCHAR[]', 'YES', None, None, None)
('countries_tags', 'VARCHAR[]', '

In [12]:
# check exact struct fields of nutriments
con.execute("SELECT nutriments[1] FROM products WHERE nutriments IS NOT NULL LIMIT 1").fetchone()

({'name': 'energy',
  'value': 333.0,
  '100g': 1393.0,
  'serving': 836.0,
  'unit': 'kcal',
  'prepared_value': None,
  'prepared_100g': None,
  'prepared_serving': None,
  'prepared_unit': None},)

In [77]:
# check available nutriment names
names = con.execute("""
    SELECT DISTINCT n.name
    FROM products,
    UNNEST(nutriments) AS t(n)
    ORDER BY n.name ASC
""").fetchall()

for n in names:
    print(n[0])

0
acide-caprylique
acides-gras-insatures
added-sugars
alcohol
alpha-linolenic-acid
arachidic-acid
arachidonic-acid
b6
behenic-acid
beta-carotene
bicarbonate
biotin
butyric-acid
caffeine
calcium
capric-acid
carbohydrates
carbohydrates-total
carbon-footprint
carbon-footprint-from-known-ingredients
carbon-footprint-from-meat-or-fish
chloride
cholesterol
choline
chromium
cocoa
copper
d-glucuronolactone
en-0
en-acide-amine-essentiel-alanine
en-acide-amine-essentiel-arginine
en-acide-amine-essentiel-aspartique
en-acide-amine-essentiel-glutamine
en-acide-amine-essentiel-glycine
en-acide-amine-essentiel-histidine
en-acide-amine-essentiel-isoleucine
en-acide-amine-essentiel-leucine
en-acide-amine-essentiel-lysine
en-cal
en-d-glucuronolactone
en-erythritol
en-fer
en-folacin
en-guarana
en-iro
en-linolenic-acid
en-panache-ginseng
en-sucralose
en-vit
en-vitamin-b3-niacin
energy
energy-from-fat
energy-kcal
energy-kj
erythritol
fat
fer
fiber
fluoride
folates
fr-d-glucuronolactone
fr-extrait-de-peau-d

In [99]:
query="SELECT DISTINCT regexp_replace(UNNEST(categories_tags), '^en:', '') AS category FROM products ORDER BY category LIMIT 10"
print(con.execute(query).fetchall())

[('100-natural-coconut-water',), ('100-per-175g',), ('12-grain-bagels',), ('120',), ('140',), ('170-calories-250-ml',), ('2',), ('210',), ('5-beer',), ('70',)]


In [89]:
# Find cereals with lowest sugar
query = """
SELECT product_name[1].text as product_name, struct_extract(n, '100g') as sugars_100g, nutriscore_grade
FROM products, UNNEST(nutriments) AS t(n)
WHERE struct_extract(n, 'name') = 'sugars'
AND array_to_string(categories_tags, ',') LIKE '%almond-butters%'
AND struct_extract(n, '100g') IS NOT NULL
ORDER BY sugars_100g ASC
LIMIT 10
"""

results = con.execute(query).fetchall()
print(f"{'Product':<45} {'Sugars/100g':>12} {'Nutriscore':>12}")
print("-" * 70)
for row in results:
    print(f"{str(row[0]):<45} {row[1]:>12.1f} {str(row[2]):>12}")

Product                                        Sugars/100g   Nutriscore
----------------------------------------------------------------------
Cashew Almond                                          2.0      unknown
Barney almond butter                                   3.0            b
Coconut Flavored Almond Butter                         6.7            c
Almond butter                                          6.7            b
Almond butter                                          6.7            b
Beurre d'amandes Crémeux                               6.7            a
Almond Hazelnut Butter                                 6.7            a


In [100]:
# Check for duplicate entry of product
query = """
SELECT product_name[1].text as product_name, code, struct_extract(n, '100g') as sugars_100g, nutriscore_grade
FROM products, UNNEST(nutriments) AS t(n)
WHERE struct_extract(n, 'name') = 'sugars'
AND array_to_string(categories_tags, ',') LIKE '%almond-butters%'
AND struct_extract(n, '100g') IS NOT NULL
ORDER BY sugars_100g ASC
LIMIT 10
"""
results = con.execute(query).fetchall()
for row in results:
    print(row)

('Cashew Almond', '0875405001347', 2.0, 'unknown')
('Barney almond butter', '53654533', 3.0, 'b')
('Coconut Flavored Almond Butter', '0051651093477', 6.666666507720947, 'c')
('Almond butter', '0808912004594', 6.666666507720947, 'b')
('Almond butter', '0061483058322', 6.6666998863220215, 'b')
("Beurre d'amandes Crémeux", '0068892236525', 6.670000076293945, 'a')
('Almond Hazelnut Butter', '0068892241253', 6.670000076293945, 'a')


In [40]:
# Get the average sugars in cereals
query = """
SELECT AVG(struct_extract(n,'100g')) AS sugars 
FROM products, UNNEST(nutriments) as t(n)
WHERE struct_extract(n,'name') = 'sugars'
AND array_to_string(categories_tags, ',') LIKE '%cereal%'
"""
result=con.execute(query).fetchone()
print("Average sugars per 100g (cereals):" , result[0])

Average sugars per 100g (cereals): 9.635549990843217


In [63]:
# Nutriscore distribution
query = """
SELECT nutriscore_grade, COUNT(*)
FROM products
GROUP BY nutriscore_grade
ORDER BY COUNT(*) DESC
"""

nutriscore_distribution=con.execute(query).fetchall()
print(f"{'NutriScore':<20} {'Distribution':<6}")
print('-' * 30)
for row in nutriscore_distribution:
    print(f"{str(row[0]):<20} {row[1]:<6.1f}")

NutriScore           Distribution
------------------------------
unknown              4704.0
e                    1283.0
d                    1182.0
c                    1121.0
a                    818.0 
b                    618.0 
not-applicable       273.0 
None                 1.0   


In [71]:
query="""SELECT product_name[1].text AS product_name, nutriscore_grade
    FROM products
    WHERE array_to_string(
    list_transform(product_name, x -> x.text), ','
) ILIKE '%q%'
    LIMIT 10"""
print(con.execute(query).fetchall())

[('Gmills hny nut cheerios sweetened whl grn oat cereal', 'd'), ('Toasted Berry Crisp, GO LEAN Cereal.', 'c'), ('GO Lean Crunch Cereal', 'c'), ('Life', 'a'), ('Harvest Crunch Granola Cereal Original', 'c'), ('Harvest Crunch Granola Cereal - Raisin Almond', 'e'), ('Croque nature', 'unknown'), ('Muesli Cereal Original', 'a'), ('Strawberry Cereal Bars', 'd'), ('Crispy Rice Cereal', 'unknown')]


In [102]:
# language count
query = """
SELECT 
    code,
    product_name[1].text as name_lang1,
    product_name[2].text as name_lang2,
    len(product_name) as language_count
FROM products
WHERE len(product_name) > 1
LIMIT 10
"""

count=con.execute(query).fetchall()
print(count)

[('0008577002786', 'Organic Vermont Maple Syrup Grade A Dark Color Robust Taste', '100 pure vermont organic maple syrup', 3), ('0011110020758', 'Imitation vanilla flavor', 'Imitation vanilla flavor', 2), ('0011110416469', 'lowfat MILK', 'lowfat MILK', 2), ('0011152156842', 'Fresh udon bowl', 'Fresh udon bowl', 2), ('0011152223131', 'Seto fumi furikake', 'Seto fumi furikake', 2), ('0011152223162', 'Assaisonnement pour riz', 'Assaisonnement pour riz', 2), ('0012009012168', "Chef d'oeuf™avec fromage sur muffin anglais", "Chef d'oeuf™avec fromage sur muffin anglais", 2), ('0012511472313', 'Organic Blue Agave', 'Organic Blue Agave', 3), ('0013087245950', ' Gâteau double chocolat ', ' Gâteau double chocolat ', 2), ('0013562000500', 'Macaroni & Cheese Classic Cheddar', 'Macaroni & Cheese Classic Cheddar', 3)]


In [105]:
#  health landscape of Canadian food products
query = """
SELECT 
    nutriscore_grade,
    COUNT(*) as count,
    ROUND(AVG(struct_extract(n, '100g')), 2) as avg_sugars
FROM products, UNNEST(nutriments) AS t(n)
WHERE struct_extract(n, 'name') = 'sugars'
AND categories_tags[1] = 'en:beverages'
AND nutriscore_grade IS NOT NULL
GROUP BY nutriscore_grade
ORDER BY nutriscore_grade
"""

results = con.execute(query).fetchall()
print(f"{'Nutriscore':<15} {'Count':>8} {'Avg Sugars/100g':>18}")
print("-" * 45)
for row in results:
    print(f"{str(row[0]):<15} {row[1]:>8} {row[2]:>18.2f}")

Nutriscore         Count    Avg Sugars/100g
---------------------------------------------
a                     19               0.00
b                     20               1.48
c                     45               1.55
d                     25               6.93
e                     63              11.48
not-applicable         6               2.91
unknown               21              32.72


In [108]:
# Get a real product code from your dataset
query = """
SELECT code, product_name[1].text, brands
FROM products
WHERE product_name[1].text IS NOT NULL
AND brands IS NOT NULL
LIMIT 10
"""
results = con.execute(query).fetchall()
for row in results:
    print(row)

('0008577002786', 'Organic Vermont Maple Syrup Grade A Dark Color Robust Taste', 'usda organic Butternut Mountain Farm,Butternut Mountain Farm')
('0011110020758', 'Imitation vanilla flavor', 'Kroger')
('0011110416469', 'lowfat MILK', 'Kroger')
('0011152223131', 'Seto fumi furikake', 'ajishima')
('0011152223162', 'Assaisonnement pour riz', 'Ajishima')
('0012009012168', "Chef d'oeuf™avec fromage sur muffin anglais", 'A&W')
('0012511472313', 'Organic Blue Agave', 'Wholesome')
('0013087245950', ' Gâteau double chocolat ', 'Oakrun Farm')
('0013562000500', 'Macaroni & Cheese Classic Cheddar', "Annie's, Annie's Homegrown")
('0013600000677', 'Truvia', 'Truvia')


In [110]:
query = """
SELECT 
    product_name[1].text as product_name,
    brands,
    nutriscore_grade,
    nova_group,
    ecoscore_grade,
    LIST(struct_extract(n, 'name') || ': ' || 
    CAST(struct_extract(n, '100g') AS VARCHAR)) as nutrients
FROM products, UNNEST(nutriments) AS t(n)
WHERE code = '0008577002786'
GROUP BY product_name, brands, nutriscore_grade, nova_group, ecoscore_grade
"""
results = con.execute(query).fetchall()

if results:
    row = results[0]
    print("=" * 50)
    print("PRODUCT SUMMARY")
    print("=" * 50)
    print(f"Product     : {row[0]}")
    print(f"Brand       : {row[1]}")
    print(f"Nutriscore  : {row[2].upper()}")
    print(f"Nova Group  : {row[3]}")
    print(f"Ecoscore    : {row[4].upper()}")
    print("-" * 50)
    print("NUTRIENTS (per 100g):")
    print("-" * 50)
    for nutrient in row[5]:
        name, value = nutrient.split(": ")
        print(f"  {name:<45} {float(value):>8.2f}")
    print("=" * 50)
else:
    print("Product not found!")

PRODUCT SUMMARY
Product     : Organic Vermont Maple Syrup Grade A Dark Color Robust Taste
Brand       : usda organic Butternut Mountain Farm,Butternut Mountain Farm
Nutriscore  : E
Nova Group  : 2
Ecoscore    : B
--------------------------------------------------
NUTRIENTS (per 100g):
--------------------------------------------------
  nutrition-score-fr                               19.00
  fruits-vegetables-nuts-estimate-from-ingredients     0.00
  sodium                                            0.01
  sugars                                           88.33
  proteins                                          0.00
  fat                                               0.00
  energy-kcal                                     333.00
  salt                                              0.02
  fruits-vegetables-legumes-estimate-from-ingredients     0.00
  carbohydrates                                    88.33
  nova-group                                        2.00
  energy                   

In [118]:
# Group by brand + product name, count variants
query = """
SELECT brands, product_name[1].text as product_name, 
COUNT(*) as variants,
LIST(code) as barcodes
FROM products
GROUP BY brands, product_name[1].text
HAVING COUNT(*) > 1
ORDER BY variants DESC
LIMIT 20
"""

results = con.execute(query).fetchall()
print(f"{'Brand':<25} {'Product':<30} {'Variants':>10}")
print("-" * 70)
for row in results:
    brand = str(row[0]) if row[0] else "MISSING"
    name = str(row[1]) if row[1] else "MISSING"
    print(f"{brand:<25} {name:<30} {row[2]:>10}")

Brand                     Product                          Variants
----------------------------------------------------------------------
MISSING                   MISSING                               290
MISSING                   MISSING                                39
MISSING                   Miel                                    8
Heinz                     Tomato Ketchup                          7
Selection                 MISSING                                 6
Great Value               MISSING                                 4
Coca-Cola                 Coca-Cola                               4
French's                  Tomato Ketchup                          4
Nestlé                    MISSING                                 4
Pepsi                     Diet Pepsi                              4
MISSING                   Yogourt                                 4
Ferrero                   Nutella                                 4
Kraft                     Smooth Peanut Butte

In [111]:
con.close()

In [ ]:
"""
From here on we have the solution for the problems

1. Missing brand/name data: 290+ products have neither brand nor name. Perfect thing to flag in your AI insights layer.
2. Real duplicates: These are the same products in different sizes/packaging — deduplication problem.
3. Missing brand OR name: any one is missing

"""

In [120]:
# Canonical Product table
con.execute("""
CREATE TABLE IF NOT EXISTS canonical_products AS
SELECT 
    ROW_NUMBER() OVER () as product_group_id,
    brands,
    (LIST_FILTER(product_name, x -> x.lang = 'en'))[1].text as name_en,
    (LIST_FILTER(product_name, x -> x.lang = 'fr'))[1].text as name_fr,
    COUNT(*) as variant_count,
    LIST(code) as barcodes,
    -- pick the most complete record as canonical
    ARG_MAX(code, completeness) as best_barcode,
    MAX(completeness) as completeness_score,
    -- use nutriscore from most complete record
    ARG_MAX(nutriscore_grade, completeness) as nutriscore_grade,
    ARG_MAX(ecoscore_grade, completeness) as ecoscore_grade,
    ARG_MAX(nova_group, completeness) as nova_group
FROM products
GROUP BY brands, name_en, name_fr
""")

count = con.execute("SELECT COUNT(*) FROM canonical_products").fetchone()
print("Canonical products:", count[0])

Canonical products: 9511


In [121]:
# Barcode mapping table
con.execute("""
CREATE TABLE IF NOT EXISTS barcode_map AS
SELECT 
    c.product_group_id,
    UNNEST(c.barcodes) as barcode,
    c.name_en,
    c.name_fr,
    c.brands
FROM canonical_products c
""")

count = con.execute("SELECT COUNT(*) FROM barcode_map").fetchone()
print("Barcode mappings:", count[0])

Barcode mappings: 10000


In [127]:
# given any barcode, find its canonical product
query = """
SELECT 
    bm.product_group_id,
    bm.brands,
    bm.name_en,
    bm.name_fr,
    cp.variant_count,
    cp.barcodes,
    cp.best_barcode,
    cp.nutriscore_grade
FROM barcode_map bm
JOIN canonical_products cp ON bm.product_group_id = cp.product_group_id
WHERE bm.barcode = '0057000002992'
"""
results = con.execute(query).fetchall()
print("Below is the canonical product for 0057000002992")
for row in results:
    print(row)

Below is the canonical product for 0057000002992
(1804, 'Heinz', 'Tomato Ketchup', None, 2, ['0057000002992', '0057000023249'], '0057000002992', 'd')


In [123]:
# Check the most deduplicated products
query = """
SELECT 
    brands,
    name_en,
    name_fr,
    variant_count,
    barcodes,
    best_barcode,
    nutriscore_grade
FROM canonical_products
WHERE variant_count > 1
ORDER BY variant_count DESC
LIMIT 10
"""
results = con.execute(query).fetchall()

print(f"{'Brand':<20} {'Name EN':<25} {'Name FR':<25} {'Variants':>10}")
print("-" * 85)
for row in results:
    brand = str(row[0]) if row[0] else "MISSING"
    name_en = str(row[1]) if row[1] else "MISSING"
    name_fr = str(row[2]) if row[2] else "MISSING"
    print(f"{brand:<20} {name_en:<25} {name_fr:<25} {row[3]:>10}")

Brand                Name EN                   Name FR                     Variants
-------------------------------------------------------------------------------------
MISSING              MISSING                   MISSING                          304
MISSING              MISSING                   MISSING                           39
Selection            MISSING                   MISSING                            9
MISSING              MISSING                   Miel                               8
Compliments          MISSING                   MISSING                            6
Great Value          MISSING                   MISSING                            4
Kirkland Signature   MISSING                   MISSING                            4
MISSING              MISSING                   Yogourt                            4
Philadelphia         MISSING                   MISSING                            4
Ferrero              MISSING                   Nutella                    

In [125]:
# Heinz ketchup barcode
query = """
SELECT cp.product_group_id, cp.brands, cp.name_en, cp.name_fr, cp.variant_count, cp.best_barcode
FROM barcode_map bm
JOIN canonical_products cp USING (product_group_id)
WHERE bm.barcode = '0057000002992'
"""
print(con.execute(query).fetchone())

(1804, 'Heinz', 'Tomato Ketchup', None, 2, '0057000002992')
